# ConformerCaller — Distributed / Google Colab Training

This notebook trains the **ConformerCaller** end-to-end model directly from raw nanopore signals using Connectionist Temporal Classification (CTC) loss.

### Training Setup
- **Architecture**: 5-layer 1D CNN subsampler (16× temporal downsampling) + 6-layer Conformer encoder (`d_model=256`, 8 attention heads, depthwise conv kernel 31)
- **Loss function**: PyTorch `CTCLoss(zero_infinity=True)` with 65 output classes (64 payload motifs + 1 CTC blank)
- **Optimisation**: AdamW (`lr=3e-4`, `weight_decay=1e-5`), 5-epoch linear warmup followed by cosine annealing, gradient clipping norm = 5.0
- **Mixed precision**: PyTorch Automatic Mixed Precision (AMP FP16)
- **Dataset**: Sharded NPZ signal streams with pre-partitioned train/validation/test split definitions (`shard_split_ids.json`)

### Recommended Runtime
- GPU: NVIDIA A100 (High RAM) or V100/T4


In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install -q python-Levenshtein tqdm
print('✓ Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.1 MB/s eta 0:00:00
✓ Done


In [ ]:
# ── Cell 2: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA = '/content/drive/MyDrive/motifcaller_data'
assert os.path.exists(DRIVE_DATA), f'Drive folder not found: {DRIVE_DATA}'

all_shards = sorted([f for f in os.listdir(DRIVE_DATA) if f.startswith('signals_shard')])
print(f'Found {len(all_shards)} shards')
assert os.path.exists(f'{DRIVE_DATA}/shard_split_ids.json'), 'Missing shard_split_ids.json'
print('shard_split_ids.json found')

Mounted at /content/drive
Found 67 shards
shard_split_ids.json found


In [ ]:
# ── Cell 3: Copy ALL shards to Colab NVMe SSD (5-10x faster than Drive I/O) ──
# This takes ~10-15 min but saves many hours of I/O during training.
import shutil
from tqdm import tqdm

LOCAL_SHARDS = '/content/local_shards'
os.makedirs(LOCAL_SHARDS, exist_ok=True)

print(f'Copying {len(all_shards)} shards to SSD...')
for shard in tqdm(all_shards):
    dst = f'{LOCAL_SHARDS}/{shard}'
    if not os.path.exists(dst):
        shutil.copy(f'{DRIVE_DATA}/{shard}', dst)

for f in ['shard_split_ids.json', 'dataset_train.pkl', 'dataset_val.pkl',
          'dataset_test.pkl', 'completed_library.csv']:
    shutil.copy(f'{DRIVE_DATA}/{f}', f'/content/{f}')

print('All data on local SSD')
!df -h /content | tail -1

Copying 67 shards to SSD...


100%|██████████| 67/67 [07:00<00:00,  6.27s/it]


All data on local SSD
overlay         236G   99G  137G  42% /


In [ ]:
# ── Cell 4: Unzip code ────────────────────────────────────────────────────────
import sys
os.makedirs('/content/motifcaller', exist_ok=True)
shutil.copy(f'{DRIVE_DATA}/motifcaller_code.zip', '/content/motifcaller_code.zip')
!cd /content && unzip -q motifcaller_code.zip -d motifcaller/
os.chdir('/content/motifcaller')
sys.path.insert(0, '/content/motifcaller')

colab_config = '''import torch
COMPLETED_LIBRARY_PATH = "/content/completed_library.csv"
PROCESSED_DATA_DIR = None; FAST5_DIR = None; MASTER_DB_PATH = None
MODEL_SAVE_DIR = "/content/checkpoints"
RESULTS_DIR = "/content/results"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_SIGNAL_LENGTH = 20_000
DEFAULT_TRAIN_CONFIG = {
    "batch_size": 16, "learning_rate": 3e-4, "weight_decay": 1e-5,
    "n_epochs": 60, "patience": 10, "grad_clip_norm": 5.0,
    "lr_scheduler": "cosine", "warmup_epochs": 5,
    "min_payload_motifs": 4, "forward_strand_only": False, "max_signal_len": 20_000,
}
N_PAYLOAD_MOTIFS = 64; N_CLASSES = 65; CTC_BLANK = 0
'''
with open('/content/motifcaller/config.py', 'w') as f:
    f.write(colab_config)
print('Code ready')

Code ready


In [ ]:
# ── Cell 5: Verify GPU ────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), 'No GPU!'
device = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

GPU: NVIDIA A100-SXM4-80GB | VRAM: 85.1GB


In [ ]:
# ── Cell 6: Load split manifest & build lookup tables ─────────────────────────
import json, gc, copy
import numpy as np
from src.data.motif_library import MotifLibrary
from src.data.dataset import MotifDatasetIndex, MotifSignalDataset, motif_collate_fn
from src.data.fast5_reader import Fast5Reader
from torch.utils.data import DataLoader

with open('/content/shard_split_ids.json') as f:
    shard_split_ids = json.load(f)

lib       = MotifLibrary('/content/completed_library.csv')
train_idx = MotifDatasetIndex.load('/content/dataset_train.pkl')
val_idx   = MotifDatasetIndex.load('/content/dataset_val.pkl')
test_idx  = MotifDatasetIndex.load('/content/dataset_test.pkl')
reader    = Fast5Reader(normalise=True)

# O(1) lookup: read_id -> index entry
train_lookup = {e[0]: e for e in train_idx.entries}
val_lookup   = {e[0]: e for e in val_idx.entries}
test_lookup  = {e[0]: e for e in test_idx.entries}

TRAIN_SHARDS = [s for s in sorted(shard_split_ids) if shard_split_ids[s]['train'] > 0]
VAL_SHARDS   = [s for s in sorted(shard_split_ids) if shard_split_ids[s]['val'] > 0]
TEST_SHARDS  = [s for s in sorted(shard_split_ids) if shard_split_ids[s]['test'] > 0]
print(f'Train shards: {len(TRAIN_SHARDS)} | Val shards: {len(VAL_SHARDS)} | Test shards: {len(TEST_SHARDS)}')
print(f'Lib: {lib} | Train: {len(train_lookup):,} | Val: {len(val_lookup):,}')

/content/motifcaller/src/data/__init__.py:3: RuntimeWarning: vbz_h5py_plugin not found. Install with: pip install vbz-h5py-plugin
FAST5 files compressed with VBZ will fail to read without this plugin.
  from .fast5_reader import Fast5Reader, mad_normalise, build_fast5_index


Train shards: 51 | Val shards: 11 | Test shards: 7
Lib: MotifLibrary(n_classes=65, payload_motifs=64, library_motifs=82) | Train: 501,330 | Val: 100,265


In [ ]:
# ── Cell 7: Helper functions ──────────────────────────────────────────────────
BATCH_SIZE = 64

def load_shard(name):
    npz = np.load(f'{LOCAL_SHARDS}/{name}', allow_pickle=False)
    return {k: npz[k] for k in npz.files}

def make_loader(shard_name, lookup, split, shuffle=True):
    ids = shard_split_ids[shard_name].get(f'{split}_ids', [])
    entries = [lookup[rid] for rid in ids if rid in lookup]
    if not entries: return None, {}
    cache = load_shard(shard_name)
    idx = copy.copy(train_idx if split == 'train' else val_idx if split == 'val' else test_idx)
    idx.entries = entries
    ds = MotifSignalDataset(idx, reader, 20_000, cache)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                        collate_fn=motif_collate_fn, num_workers=2, pin_memory=True)
    return loader, cache

# Sanity check
l, c = make_loader(TRAIN_SHARDS[0], train_lookup, 'train')
b = next(iter(l))
print(f'Shard check: sigs={b[0].shape}, labels={b[1].shape}')
del l, c, b; gc.collect()

Shard check: sigs=torch.Size([64, 1, 20000]), labels=torch.Size([338])


891

In [ ]:
# ── Cell 7.5: Full Pipeline Dry Run ──────────────────────────────────────────
from src.model.motifcaller import build_model
import torch.nn as nn
import gc

print("--- Initialising Dry Run ---")
test_model = build_model('conformer', n_classes=lib.n_classes).to(device)
test_ctc = nn.CTCLoss(blank=0, zero_infinity=True).to(device)

# Force eval mode and disable gradients to prevent memory hoarding during the test
test_model.eval()

for split_name, shard_list, lookup_dict in [('train', TRAIN_SHARDS, train_lookup), ('val', VAL_SHARDS, val_lookup)]:
    if not shard_list:
        print(f"⚠ No {split_name} shards found.")
        continue

    target_shard = shard_list[0]
    print(f"Testing {split_name.upper()} pipeline on {target_shard}...")

    test_loader, test_cache = make_loader(target_shard, lookup_dict, split_name)
    test_batch = next(iter(test_loader))
    sigs, flat_lbl, sig_lens, lbl_lens = test_batch

    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            log_probs = test_model(sigs.to(device), sig_lens.to(device))
            test_loss = test_ctc(
                log_probs,
                flat_lbl.to(device),
                test_model.output_lengths(sig_lens).to(device),
                lbl_lens.to(device)
            )

    print(f"✓ {split_name.capitalize()} pass successful. Sig shape: {sigs.shape} | Loss: {test_loss.item():.4f}")

    del test_loader, test_cache, test_batch; gc.collect(); torch.cuda.empty_cache()

del test_model, test_ctc; gc.collect(); torch.cuda.empty_cache()
print("--- Dry Run Complete ---")

--- Initialising Dry Run ---
Testing TRAIN pipeline on signals_shard_0000.npz...
✓ Train pass successful. Sig shape: torch.Size([16, 1, 20000]) | Loss: 467.1350
Testing VAL pipeline on signals_shard_0050.npz...
✓ Val pass successful. Sig shape: torch.Size([16, 1, 20000]) | Loss: 423.5915
--- Dry Run Complete ---


In [ ]:
# ── Cell 8: Build model ───────────────────────────────────────────────────────
from src.model.motifcaller import build_model, count_parameters
ARCH = 'conformer'
N_EPOCHS = 60
LR = 3e-4
model = build_model(ARCH, n_classes=lib.n_classes).to(device)
print(f'Model: {ARCH} | {count_parameters(model)/1e6:.2f}M params')

Model: conformer | 11.02M params


In [ ]:
# ── Cell 9: Training loop ─────────────────────────────────────────────────────
import torch.nn as nn, torch.optim as optim, math, time, json, random
from pathlib import Path

DRIVE_CKPT = Path('/content/drive/MyDrive/motifcaller_checkpoints')
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
ckpt_path = DRIVE_CKPT / f'{ARCH}_best_model.pt'

scaler     = torch.amp.GradScaler('cuda')
ctc        = nn.CTCLoss(blank=0, zero_infinity=True).to(device)
optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
best_val   = float('inf')
start_ep   = 0
pat_count  = 0
PATIENCE   = 7
history    = []

def lr_schedule(ep, warmup=5, total=N_EPOCHS, base=LR):
    if ep < warmup: return base * (ep + 1) / warmup
    return base * 0.5 * (1 + math.cos(math.pi * (ep-warmup)/max(1,total-warmup)))

if ckpt_path.exists():
    c = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(c['model']); optimizer.load_state_dict(c['optimizer'])
    start_ep = c['epoch'] + 1; best_val = c['best_val_loss']
    if best_val == 0.0: best_val = float('inf')
    history = c.get('history', [])
    print(f'Resumed epoch {start_ep}, best_val={best_val:.4f}')

for epoch in range(start_ep, N_EPOCHS):
    for g in optimizer.param_groups: g['lr'] = lr_schedule(epoch)
    t0 = time.time()

    # Training
    model.train(); train_losses = []
    for shard_name in random.sample(TRAIN_SHARDS, len(TRAIN_SHARDS)):
        loader, cache = make_loader(shard_name, train_lookup, 'train')
        if loader is None: continue
        shard_l = []
        for batch in loader:
            if batch is None: continue
            sigs, flat_lbl, sig_lens, lbl_lens = batch
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                lp = model(sigs.to(device), sig_lens.to(device))
                loss = ctc(lp, flat_lbl.to(device), model.output_lengths(sig_lens).to(device), lbl_lens.to(device))
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer); scaler.update()
            shard_l.append(loss.item())
        train_losses.extend(shard_l)
        avg = sum(shard_l)/max(1,len(shard_l))
        print(f'  E{epoch} {shard_name} n={shard_split_ids[shard_name]["train"]} loss={avg:.4f}', flush=True)
        del cache, loader; gc.collect(); torch.cuda.empty_cache()

    # Validation — ALL val shards, guaranteed no inf
    model.eval(); val_losses = []
    with torch.no_grad():
        for shard_name in VAL_SHARDS:
            loader, cache = make_loader(shard_name, val_lookup, 'val', shuffle=False)
            if loader is None: continue
            for batch in loader:
                if batch is None: continue
                sigs, flat_lbl, sig_lens, lbl_lens = batch
                with torch.amp.autocast('cuda'):
                    lp = model(sigs.to(device), sig_lens.to(device))
                    loss = ctc(lp, flat_lbl.to(device), model.output_lengths(sig_lens).to(device), lbl_lens.to(device))
                if torch.isfinite(loss): val_losses.append(loss.item())
            del cache, loader; gc.collect()

    avg_train = sum(train_losses)/max(1,len(train_losses))
    avg_val   = sum(val_losses)/max(1,len(val_losses)) if val_losses else float('inf')
    print(f'★ E{epoch:3d} train={avg_train:.4f} val={avg_val:.4f} lr={optimizer.param_groups[0]["lr"]:.1e} {(time.time()-t0)/60:.1f}min', flush=True)
    history.append({'epoch': epoch, 'train': avg_train, 'val': avg_val if avg_val != float('inf') else None})

    if avg_val < best_val:
        best_val = avg_val; pat_count = 0
        torch.save({'epoch': epoch, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'best_val_loss': best_val, 'arch': ARCH, 'n_classes': lib.n_classes, 'history': history}, ckpt_path)
        print(f'  Best saved (val={best_val:.4f})', flush=True)
    else:
        pat_count += 1
        if pat_count >= PATIENCE: print(f'Early stop (patience={PATIENCE})'); break

    with open(str(DRIVE_CKPT / 'training_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

print(f'Done. Best val: {best_val:.4f}')

Resumed epoch 47, best_val=0.7992
  E47 signals_shard_0040.npz n=10000 loss=0.7281
  E47 signals_shard_0038.npz n=10000 loss=0.7230
  E47 signals_shard_0031.npz n=10000 loss=0.7232
  E47 signals_shard_0045.npz n=10000 loss=0.7183
  E47 signals_shard_0005.npz n=10000 loss=0.7205
  E47 signals_shard_0008.npz n=10000 loss=0.7222
  E47 signals_shard_0026.npz n=10000 loss=0.7317
  E47 signals_shard_0034.npz n=10000 loss=0.7187
  E47 signals_shard_0024.npz n=10000 loss=0.7321
  E47 signals_shard_0027.npz n=10000 loss=0.7220
  E47 signals_shard_0001.npz n=10000 loss=0.7161
  E47 signals_shard_0011.npz n=10000 loss=0.7154
  E47 signals_shard_0050.npz n=1330 loss=0.6889
  E47 signals_shard_0016.npz n=10000 loss=0.7263
  E47 signals_shard_0046.npz n=10000 loss=0.7360
  E47 signals_shard_0007.npz n=10000 loss=0.7222
  E47 signals_shard_0025.npz n=10000 loss=0.7124
  E47 signals_shard_0000.npz n=10000 loss=0.7214
  E47 signals_shard_0010.npz n=10000 loss=0.7193
  E47 signals_shard_0012.npz n=10000

In [ ]:
# ── Cell 10: Plot training curves ─────────────────────────────────────────────
import matplotlib.pyplot as plt
with open(str(DRIVE_CKPT / 'training_history.json')) as f: history = json.load(f)
epochs = [h['epoch'] for h in history]
train_l = [h['train'] for h in history]
val_l = [h['val'] for h in history if h['val'] is not None]
val_ep = [h['epoch'] for h in history if h['val'] is not None]
plt.figure(figsize=(10,4))
plt.plot(epochs, train_l, label='Train'); plt.plot(val_ep, val_l, 'o-', label='Val')
plt.xlabel('Epoch'); plt.ylabel('CTC Loss'); plt.title(f'Motifcaller {ARCH}')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
shutil.copy('/content/training_curves.png', str(DRIVE_CKPT / 'training_curves.png'))
plt.show()

In [ ]:
# ── Cell 11: Download checkpoint ──────────────────────────────────────────────
from google.colab import files
files.download(str(ckpt_path))
files.download(str(DRIVE_CKPT / 'training_history.json'))
files.download('/content/training_curves.png')